In [2]:
import requests
import pandas as pd

# Use IBM's static JSON URL (the live API often fails in Colab)
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

response = requests.get(static_json_url)
print("Status code:", response.status_code)

# Convert to DataFrame
data = pd.json_normalize(response.json())
print("Number of launches:", len(data))
print(data.head())

Status code: 200
Number of launches: 107
       static_fire_date_utc  static_fire_date_unix    tbd    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False  False     0.0   
1                      None                    NaN  False  False     0.0   
2                      None                    NaN  False  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False  False     0.0   
4                      None                    NaN  False  False     0.0   

                     rocket  success  \
0  5e9d0d95eda69955f709d1eb    False   
1  5e9d0d95eda69955f709d1eb    False   
2  5e9d0d95eda69955f709d1eb    False   
3  5e9d0d95eda69955f709d1eb     True   
4  5e9d0d95eda69955f709d1eb     True   

                                             details crew ships  ...  \
0   Engine failure at 33 seconds and loss of vehicle   []    []  ...   
1  Successful first stage burn and transition to ...   []    []  ...   
2  Residual stage 1 thrust led to collision b

In [5]:
data = response.json()

records = []
for launch in data:
    record = {
        'FlightNumber': launch.get('flight_number'),
        'Date': launch.get('date_utc'),
        'BoosterVersion': launch.get('rocket'),
        'PayloadMass': None,
        'Orbit': None,
        'LaunchSite': launch.get('launchpad'),
        'Outcome': None,
        'Flights': None,
        'GridFins': None,
        'Reused': None,
        'Legs': None,
        'LandingPad': None,
        'Block': None,
        'ReusedCount': None,
        'Serial': None,
    }

    # payloads is a list of IDs (strings) in this dataset
    if launch.get('payloads'):
        p = launch['payloads'][0]
        if isinstance(p, dict):
            record['PayloadMass'] = p.get('mass_kg')
            record['Orbit'] = p.get('orbit')

    # cores is a list of IDs (strings) too
    if launch.get('cores'):
        c = launch['cores'][0]
        if isinstance(c, dict):
            record['Outcome'] = c.get('landing_success')
            record['Flights'] = c.get('flight')
            record['GridFins'] = c.get('gridfins')
            record['Reused'] = c.get('reused')
            record['Legs'] = c.get('legs')
            record['LandingPad'] = c.get('landpad')
            record['Block'] = c.get('block')
            record['ReusedCount'] = c.get('reuse_count')
            record['Serial'] = c.get('core')

    records.append(record)

df = pd.DataFrame(records)
print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

Shape: (107, 15)
   FlightNumber                      Date            BoosterVersion  \
0             1  2006-03-24T22:30:00.000Z  5e9d0d95eda69955f709d1eb   
1             2  2007-03-21T01:10:00.000Z  5e9d0d95eda69955f709d1eb   
2             3  2008-08-03T03:34:00.000Z  5e9d0d95eda69955f709d1eb   
3             4  2008-09-28T23:15:00.000Z  5e9d0d95eda69955f709d1eb   
4             5  2009-07-13T03:35:00.000Z  5e9d0d95eda69955f709d1eb   

  PayloadMass Orbit                LaunchSite Outcome  Flights  GridFins  \
0        None  None  5e9e4502f5090995de566f86    None        1     False   
1        None  None  5e9e4502f5090995de566f86    None        1     False   
2        None  None  5e9e4502f5090995de566f86    None        1     False   
3        None  None  5e9e4502f5090995de566f86    None        1     False   
4        None  None  5e9e4502f5090995de566f86    None        1     False   

   Reused   Legs LandingPad Block ReusedCount                    Serial  
0   False  False       No

In [6]:
df.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv")
print("Rows:", len(df))

Saved dataset_part_1.csv
Rows: 107
